# FewDSR — Colab T4 Execution Runbook (Part 2 & Part 3)

This notebook executes `docs/RUNBOOK_PART2.md` and `docs/RUNBOOK_PART3.md` on a Colab T4 GPU with real TORGO data.

**Before running:**
1. Select **Runtime → Change runtime type → T4 GPU**.
2. Upload `fewdsr/` (or clone your repo) to `/content/fewdsr`, or mount Google Drive.
3. Have the TORGO dataset available (either in Google Drive or downloaded to `/content/TORGO`).

In [ ]:
!nvidia-smi
!python --version

## 1. Workspace & Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# the folder that CONTAINS requirements.txt (delete any .venv inside it before uploading)
%cd /content/drive/MyDrive/fewdsr


In [ ]:
!pip install -q -r requirements.txt
!pip uninstall -y -q torchao
!pip list 2>/dev/null | grep -iE "^(torch|transformers|peft|ctranslate2|faster-whisper|scipy) "


## 2. Part 2 Verification (Tests, LoRA & Backend Checks)

In [ ]:
!python -m unittest discover -s tests
!FEWDSR_SLOW=1 python -m unittest tests.test_part2_slow -v

In [ ]:
!python -m experiments.check_lora
!python -m experiments.check_backend

## 3. Prepare TORGO manifest (Hugging Face dataset `abnerh/TORGO-database`)
Downloads the dataset (~1.5 GB, short words + restricted sentences only), writes the wav files of the **dysarthric** speakers (head mic) to local disk `/content/torgo_wav`, and builds `data/manifest.json`.
Re-run this cell after a runtime reset (wav files live on the ephemeral disk). Read the printed table: speakers whose list of feasible K lacks a budget will skip that budget (reported, never faked).


In [ ]:
!python -m experiments.prepare_data_hf --wav-dir /content/torgo_wav --out data/manifest.json


## 4. Part 2 Smoke Run (2-3 minutes)

In [ ]:
!python -m experiments.run_baseline --speakers F01 --limit 3 --out results/smoke --force
!python -m experiments.run_adapt    --speakers F01 --budgets 5 --epochs 2 --limit 3 --out results/smoke --adapters artifacts/smoke --force
!python -m experiments.validate_raw results/smoke


## 5. Part 2 Real Runs
Verify the severity map in `fewdsr_config.py` against the official TORGO documentation. Run at least one mild (`F04`, `F03`, `M03`) and one severe (`M01`, `F01`, `M02`, `M04`) speaker — or all 8 speakers if time permits.

In [ ]:
SPEAKERS = "F04 M01"  # or e.g. "F01 F03 F04 M01 M02 M03 M04 M05"
# --force: re-run everything (the manifest and the LoRA training code changed after the first attempt)
!python -m experiments.run_baseline --speakers {SPEAKERS} --max-test 100 --force
!python -m experiments.run_adapt    --speakers {SPEAKERS} --budgets 5 10 20 40 --max-test 100 --force
!python -m experiments.validate_raw results/raw


## 6. Part 3: Calibration, Silence Stress Test & INT8 Edge Benchmark

In [ ]:
!python -m experiments.run_calibration --raw results/raw --out results/calibration.json

In [ ]:
!python -m experiments.run_silence_stress --manifest data/manifest.json --speakers {SPEAKERS} --n 10

In [ ]:
!python -m experiments.run_edge --check
!python -m experiments.run_edge --manifest data/manifest.json --speaker F04 --k 10 --n 30 --adapter artifacts/adapters/F04_K10

## 7. Part 3: Summary, Figures, Report & Acceptance Checks

In [ ]:
!python -m experiments.run_all --raw results/raw --out results
!python -m experiments.validate_summary results/summary.json
!python -m experiments.make_report

In [ ]:
# Acceptance grep check: only docs/INTEGRITY_NOTES.md may match
!grep -rnE "36\.4|62\.5|66\.7|150\.0|0\.284|0\.082" --include=*.py --include=*.md .

## 8. Package Results for Download
Download `fewdsr_real_results.zip` and append the measured numbers, library versions, hardware, and warnings to `docs/HANDOFF_PART3.md`.

In [ ]:
!zip -r /content/fewdsr_real_results.zip results/ artifacts/adapters/ data/manifest.json
from google.colab import files
files.download('/content/fewdsr_real_results.zip')